In [14]:
import torch
import matplotlib.pyplot as plt
import math
import scipy.io
import numpy as np
from langevinDynamics.core import compute_log_density_gradient, SimulationParameters, sample_gmm_efficient, computeMasks
torch.manual_seed(42)

# Set device
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
Nref = 1*10**7       # Reduced for performance

mat = scipy.io.loadmat('data/initialData.mat')['initialData']
centers = torch.from_numpy(mat)

In [16]:
# ─── GMM for X0 (initial distribution) ─────────────────────────
std0 = math.sqrt(1.0)
weightsX0 = torch.tensor([1.0], device=device)
mu_X0 = torch.tensor([[0.,0.]], device=device)
covariances_X0 = torch.tensor([[[std0**2,0.],[0.,std0**2]]], device=device)

# ─── GMM for Xinf (final/target distribution) ──────────────────
weightsXinf = torch.tensor([0.0], device=device)
mu_Xinf = torch.tensor([[0.,0.]], device=device)
inv_covariances_T = torch.zeros_like(torch.eye(2, device=device).unsqueeze(0))
det_T = 0.0

In [ ]:
randonParticleSets = 50
testRadii = np.array([0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005])
errors = np.zeros_like(testRadii)
for (testRadius, i) in enumerate(testRadii):
    positions = sample_gmm_efficient(weightsX0, mu_X0, covariances_X0, Nref)
    error = 0.0
    for (particleSet, j) in enumerate(range(randonParticleSets)):
        
        mask = computeMasks(positions, centers, testRadius)
        driftOfCircles = compute_log_density_gradient(centers, weightsXinf, mu_Xinf, inv_covariances_T, det_T, True, device)
        particlesPerCircle = mask.sum(dim=0)
        
        # Update position of circles
        for center in range(centers.shape[0]):
            if particlesPerCircle[center].item() > 0:
                shifts = positions[mask[:, center], :] - centers[center, :]
                meanShiftVector = shifts.mean(dim=0)
                error += driftOfCircles[center, :] - 4*meanShiftVector/(testRadius**2)
            else:
                error += driftOfCircles[center, :]
    errors[i] = error


RuntimeError: expected scalar type Double but found Float